# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [17]:

# %%
import re, json, math, subprocess, tempfile, os
import pandas as pd
from pathlib import Path
from huggingface_hub import HfFileSystem
from itertools import permutations
from enum import Enum
from typing import Optional
import random


In [18]:
fs = HfFileSystem()

In [19]:
OUTPUT_FILE = Path("perturbation_pairs.jsonl")
print("Ready.")

# %%
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)
df = df[0:10]
df.head()

# %%
df = df[['signature', 'type']]
df.head()


Ready.


,signature,type
0,(A : Hopf_ C) :\n A.X.comul.hom ≫\n A.X...,∀ {C : Type u₁} [inst : CategoryTheory.Categor...
1,{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,∀ {C : Type u} [inst : CategoryTheory.Category...
2,{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,∀ {C : Type u} [inst : CategoryTheory.Category...
3,{P : C} {ι ι' : P ⟶ X} (w : ι ≫ f = ι ≫ g) (w...,{C : Type u} →\n [inst : CategoryTheory.Categ...
4,(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,∀ {C : Type u} [inst : CategoryTheory.Category...


In [20]:
# %%
_COMPILE_TEMPLATE = """\
import Mathlib
import Aesop

theorem variant_check {sig} : {typ} := by
  first
  | decide | omega | aesop | norm_num | tauto
  | sorry
"""

_NEGATION_TEMPLATE = """\
import Mathlib
import Aesop

theorem negation_check {sig} : ¬({typ}) := by
  first
  | decide | omega | aesop | norm_num | tauto
"""


def _run_lean(lean_code: str, timeout: int = 10) -> tuple[bool, str]:
    """Write lean_code to a temp file, run it, return (success, output)."""
    with tempfile.NamedTemporaryFile(suffix=".lean", mode="w", delete=False) as f:
        f.write(lean_code)
        tmp_path = f.name
    try:
        result = subprocess.run(
            ["lake", "env", "lean", tmp_path],
            capture_output=True, text=True, timeout=timeout,
        )
        return result.returncode == 0, result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        return False, "timeout"
    finally:
        os.unlink(tmp_path)


def compile_lean(variant_sig: str, variant_type: str, timeout: int = 10) -> bool:
    """Return True iff the variant statement type-checks in Lean (sorry allowed)."""
    code = _COMPILE_TEMPLATE.format(sig=variant_sig, typ=variant_type)
    success, _ = _run_lean(code, timeout=timeout)
    return success

In [21]:
# We propagate truth values symbolically through the perturbation path.
# This avoids Lean calls wherever the logic is deterministic.
#
# Propagation rules:
#   negation:               TRUE→FALSE,  FALSE→TRUE,    UNKNOWN→UNKNOWN
#   contrapositive:         TRUE→TRUE,   FALSE→FALSE,   UNKNOWN→UNKNOWN  (logically equivalent)
#   converse:               TRUE→UNKNOWN, FALSE→UNKNOWN, UNKNOWN→UNKNOWN  (not equivalent)
#   specialization:         TRUE→TRUE,   FALSE→UNKNOWN, UNKNOWN→UNKNOWN  (subset preserves truth)
#   generalization:         TRUE→UNKNOWN, FALSE→FALSE,  UNKNOWN→UNKNOWN  (superset preserves falsity)
#   bound_perturbation_looser:  TRUE→TRUE,   FALSE→UNKNOWN, UNKNOWN→UNKNOWN
#   bound_perturbation_tighter: TRUE→UNKNOWN, FALSE→FALSE,  UNKNOWN→UNKNOWN

# %%
class TruthValue(Enum):
    TRUE    = "true"
    FALSE   = "false"
    UNKNOWN = "unknown"


# Propagation table: perturbation_name -> {current_truth -> new_truth}
_PROPAGATION_RULES: dict[str, dict[TruthValue, TruthValue]] = {
    "negation": {
        TruthValue.TRUE:    TruthValue.FALSE,
        TruthValue.FALSE:   TruthValue.TRUE,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "contrapositive": {
        TruthValue.TRUE:    TruthValue.TRUE,
        TruthValue.FALSE:   TruthValue.FALSE,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "converse": {
        TruthValue.TRUE:    TruthValue.UNKNOWN,
        TruthValue.FALSE:   TruthValue.UNKNOWN,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "specialization": {
        TruthValue.TRUE:    TruthValue.TRUE,
        TruthValue.FALSE:   TruthValue.UNKNOWN,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "generalization": {
        TruthValue.TRUE:    TruthValue.UNKNOWN,
        TruthValue.FALSE:   TruthValue.FALSE,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "bound_perturbation_looser": {
        TruthValue.TRUE:    TruthValue.TRUE,
        TruthValue.FALSE:   TruthValue.UNKNOWN,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
    "bound_perturbation_tighter": {
        TruthValue.TRUE:    TruthValue.UNKNOWN,
        TruthValue.FALSE:   TruthValue.FALSE,
        TruthValue.UNKNOWN: TruthValue.UNKNOWN,
    },
}


In [22]:

def _apply_perturbation_logic(
    current_truth: TruthValue,
    perturbation: str,
) -> TruthValue:
    """Apply one step of symbolic truth propagation."""
    rules = _PROPAGATION_RULES.get(perturbation)
    if rules is None:
        return TruthValue.UNKNOWN
    return rules[current_truth]


def infer_truth_from_path(
    anchor_truth: TruthValue,
    perturbation_path: list[str],
) -> TruthValue:
    """
    Propagate anchor truth symbolically through the full perturbation chain.
    Returns the inferred TruthValue — no Lean calls needed.
    """
    current = anchor_truth
    for perturbation in perturbation_path:
        current = _apply_perturbation_logic(current, perturbation)
    return current

In [23]:
def _lean_truth_check(
    variant_sig: str,
    variant_type: str,
    timeout: int = 10,
) -> TruthValue:
    """
    Ask Lean directly: provable (TRUE), negation provable (FALSE), or UNKNOWN.
    Used only as a fallback when symbolic inference returns UNKNOWN.
    """
    # Try to prove the statement
    code = _COMPILE_TEMPLATE.format(sig=variant_sig, typ=variant_type)
    success, output = _run_lean(code, timeout=timeout)
    if success and "sorry" not in output:
        return TruthValue.TRUE

    # Try to prove the negation
    neg_code = _NEGATION_TEMPLATE.format(sig=variant_sig, typ=variant_type)
    neg_success, _ = _run_lean(neg_code, timeout=timeout)
    if neg_success:
        return TruthValue.FALSE

    return TruthValue.UNKNOWN

In [24]:
# Strategy:
#   1. Propagate symbolically through perturbation_path (free, instant).
#   2. If still UNKNOWN, fall back to Lean tactic probes.
#   3. Return (truth_value_str, source_str).

# %%
# Anchors are assumed TRUE (they come from Mathlib — all proven theorems).
_ANCHOR_TRUTH = TruthValue.TRUE


def is_true(
    perturbation_path: list[str],
    variant_sig: str,
    variant_type: str,
    use_lean_fallback: bool = True,
    timeout: int = 10,
) -> str:
    """
    Determine truth of a variant given its perturbation path from a TRUE anchor.

    Returns one of: "true", "false", "unknown"

    Logic-first: symbolic propagation through the perturbation path is attempted
    before any Lean call. Lean is only invoked when the path leaves truth UNKNOWN.
    """
    inferred = infer_truth_from_path(_ANCHOR_TRUTH, perturbation_path)

    if inferred != TruthValue.UNKNOWN:
        return inferred.value

    if use_lean_fallback:
        lean_result = _lean_truth_check(variant_sig, variant_type, timeout)
        return lean_result.value

    return TruthValue.UNKNOWN.value


In [25]:

# %%
def apply_perturbation_chains(
    df: pd.DataFrame,
    transforms: dict,
    n_permutations: int = 6,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(random_seed)
    transform_names = list(transforms.keys())
    all_permutations = list(permutations(transform_names))

    results = []

    for _, row in df.iterrows():
        anchor_sig  = row["signature"]
        anchor_type = row["type"]

        k = min(n_permutations, len(all_permutations))
        sampled_permutations = rng.sample(all_permutations, k)

        for perm in sampled_permutations:
            current_sig  = anchor_sig
            current_type = anchor_type
            applied_so_far = []

            for transform_name in perm:
                fn = transforms[transform_name]

                result = fn(current_sig, current_type)
                if result is None:
                    break

                variant_sig, variant_type = result
                if (variant_sig.strip() == current_sig.strip()
                        and variant_type.strip() == current_type.strip()):
                    break

                if not compile_lean(variant_sig, variant_type):
                    break

                # Compiled — save this intermediate as a datapoint
                applied_so_far.append(transform_name)
                results.append({
                    **{k: v for k, v in row.items() if k not in ("signature", "type")},
                    "anchor_signature":      anchor_sig,
                    "anchor_type":           anchor_type,
                    "variant_signature":     variant_sig,
                    "variant_type":          variant_type,
                    "perturbations_applied": list(applied_so_far),
                    "chain_depth":           len(applied_so_far),
                    # is_true: symbolic inference first, Lean fallback if UNKNOWN
                    "is_true": is_true(
                        list(applied_so_far),
                        variant_sig,
                        variant_type,
                    ),
                })

                current_sig  = variant_sig
                current_type = variant_type

    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df):,} compiled variants from {len(df):,} anchors")
    if len(result_df):
        print(result_df["chain_depth"].value_counts().sort_index().to_string())
    return result_df